In [3]:
import os
import sys
p = os.path.abspath('../..')
if p not in sys.path:
    sys.path.append(p)

from numcodecs import Blosc
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import glob
import json
import zarr
import waveorder as wo
from recOrder.recOrder.compute.QLIPP_compute import reconstruct_QLIPP_3D, initialize_reconstructor, load_bg
from recOrder.recOrder.io.reader import MicromanagerReader

In [4]:
def init_zarr_store(data_path, chunk_size=(1, 65, 2048, 2048), data_shape = (10,65,2048,2448)):
    compressor=Blosc(cname='zstd', clevel=1, shuffle=Blosc.SHUFFLE)
    
    if not data_path.endswith('.zarr'):
        data_path += '.zarr'
    
    if not os.path.exists(data_path):
        os.makedirs(data_path)
        
    zarr_store = zarr.open(data_path, mode='a')
    
    
    datasets = ['Phase3D', 'Retardance', 'Orientation', 'BF']
    groups = ['RSV48', 'RSV24', 'Mock48', 'Mock24']
    
    for group in groups:
        zarr_store.create_group(group)
        for ds in datasets:
            zarr_store[group].zeros(ds, shape=data_shape, chunks=chunk_size, dtype='float32',
                                        compressor=compressor)
            
    return zarr_store

In [ ]:
data_path = ''
zarr_store = init_zarr_store(data_path, chunk_size=(1, 65, 2048, 2048), data_shape = (10,65,2048,2048))

## Gather Data Paths

In [5]:
data_folder = '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_25_40X_04NA_A549/'

sample_paths = glob.glob(data_folder+'*')
sample_paths.pop(-1)

'/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_25_40X_04NA_A549/Calibration'

In [6]:
data = MicromanagerReader(sample_paths[1]+'/_1', extract_data=True)

In [11]:
data._positions

{0: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 1: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 2: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 3: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 4: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 5: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 6: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 7: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 8: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>,
 9: <zarr.core.Array (6, 65, 2048, 2048) uint16 read-only>}

## Initialize Reconstruction

In [4]:
## Set Reconstruction Parameters
image_dim = (2048,2448)
wavelength = 525
swing = 0.05
N_channel = 4
NA_obj = 1.1
NA_illu = 0.4
mag = 40
N_slices = 65
z_step = 0.25
pad_z = 0
pixel_size = 6.5
bg_option = 'local_fit'
n_media = 1.33

reconstructor = initialize_reconstructor(image_dim, wavelength, swing, N_channel, NA_obj, NA_illu, mag, N_slices, z_step, pad_z, 
                                         pixel_size, bg_option = bg_option, n_media = n_media, use_gpu=False, gpu_id = 0)



Initializing Reconstructor...
Finished Initializing Reconstructor (1.59 min)


## Reconstruct Batch

In [7]:
zarr_store = zarr.open('/gpfs/CompMicro/projects/A549/2021_02_18_40x_04NA_A549/SlideA.zarr')

In [8]:
zarr_store.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, name='Mock24', nodes=(Node(disabled=True,…

In [6]:
zarr_store.info

Name,/Mock24/Phase3D
Type,zarr.core.Array
Data type,float32
Shape,"(10, 65, 2048, 2448)"
Chunk shape,"(1, 65, 2048, 2448)"
Order,C
Read-only,False
Compressor,"Blosc(cname='zstd', clevel=1, shuffle=SHUFFLE, blocksize=0)"
Store type,zarr.storage.DirectoryStore
No. bytes,13035110400 (12.1G)
No. bytes stored,393


In [5]:
bg_path = '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_18_40x_04NA_A549/SlideA/BG/'
bg_data = load_bg(bg_path, 2048, 2048, ROI=(416,845,645,535), cropped=True)

In [ ]:
condition = 'RSV48'

In [ ]:
for pos in range(len(data._positions)):
    ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(data._positions[pos], bg_data, reconstructor, method = "Tikhonov",
                                                                   reg_re = 1e-4, reg_im = 1e-4, rho = 1e-5, 
                                                                   lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)
        
    zarr_store[condition]['BF'][pos] = BF_stack
    zarr_store[condition]['Retardance'][pos] = ret_stack
    zarr_store[condition]['Orientation'][pos] = ori_stack
    zarr_store[condition]['Phase3D'][pos] = np.transpose(phase3D,(2,0,1))
    
    

Computing Birefringence...
Finished Computing Birefringence (1.49 min)
Computing 3d Phase...
Finished Reconstruction (2.04 min)

Computing Birefringence...
Finished Computing Birefringence (1.48 min)
Computing 3d Phase...
Finished Reconstruction (2.04 min)

Computing Birefringence...
Finished Computing Birefringence (1.50 min)
Computing 3d Phase...
Finished Reconstruction (2.04 min)

